# 🧠 Multimodal RAG System — Demo Notebook

This notebook walks through the complete pipeline:
1. Document ingestion (PDF + text + images)
2. Embedding & vector indexing with FAISS
3. Semantic retrieval
4. Answer generation
5. Evaluation with RAG metrics


In [ ]:
# Install dependencies
!pip install sentence-transformers faiss-cpu PyPDF2 transformers openai streamlit -q

In [ ]:
import sys
sys.path.append('..')

from src.rag_pipeline import MultimodalRAGPipeline, DocumentIngester, VectorStore
from src.evaluation import RAGEvaluator

print('✅ Imports OK')

## 1. Create a Sample Document

In [ ]:
import os
os.makedirs('../data/sample', exist_ok=True)

# Create a sample text document
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval
with language model generation. Instead of relying solely on parametric knowledge stored
in model weights, RAG retrieves relevant documents from an external knowledge base and
conditions the generation on those documents.

The core components of a RAG system are:
1. Document ingestion: loading and chunking documents into manageable pieces
2. Embedding: converting text into dense vector representations
3. Vector store: an index for efficient similarity search (e.g., FAISS)
4. Retrieval: finding the most similar chunks to a given query
5. Generation: using an LLM to produce an answer conditioned on retrieved context

Multimodal RAG extends this by handling not just text but also images, tables, and other
data modalities. Images are typically processed via captioning models (e.g., BLIP)
before being embedded alongside textual content.

Evaluation of RAG systems typically involves metrics such as:
- Faithfulness: is the answer grounded in the retrieved context?
- Answer relevance: does the answer address the user's question?
- Context precision: are the retrieved chunks actually relevant?
"""

with open('../data/sample/rag_overview.txt', 'w') as f:
    f.write(sample_text)

print('Sample document created!')

## 2. Build the RAG Pipeline

In [ ]:
pipeline = MultimodalRAGPipeline(
    embedding_model   = 'all-MiniLM-L6-v2',
    generator_backend = 'extractive',   # Change to 'openai' with OPENAI_API_KEY
    top_k             = 3,
    chunk_size        = 100,
    chunk_overlap     = 20,
)

pipeline.ingest('../data/sample/')
print(pipeline)

## 3. Query the System

In [ ]:
questions = [
    'What are the core components of a RAG system?',
    'How are images handled in multimodal RAG?',
    'What metrics are used to evaluate RAG systems?',
]

for q in questions:
    response = pipeline.query(q)
    print(f'\nQ: {q}')
    print(f'A: {response.answer}')
    print(f'Sources: {[r.document.source for r in response.sources]}')

## 4. Evaluate the Pipeline

In [ ]:
evaluator = RAGEvaluator()
response  = pipeline.query('What is retrieval-augmented generation?')

report = evaluator.evaluate(
    query          = response.query,
    answer         = response.answer,
    context        = response.context,
    retrieved_docs = [r.document.content for r in response.sources],
    scores         = [r.score for r in response.sources],
)

evaluator.print_report(report)

## 5. Save & Reload the Index

In [ ]:
pipeline.save('../data/rag_store')

# Load in a new instance
pipeline2 = MultimodalRAGPipeline()
pipeline2.load('../data/rag_store')
r = pipeline2.query('What is FAISS?')
print('Loaded pipeline answer:', r.answer)